# 05 - Learned Embeddings: Word2Vec (CBOW, Skip-gram) & fastText (Parts B, C)
**Owner:** Omais Rizvi

Models are trained once by `src/train_embeddings.py` and saved in `models/`. Here we only LOAD them.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
df = pd.read_csv(ROOT / "data/processed/cleaned.csv").dropna(subset=["clean_text"])
docs, labels = df["clean_text"].tolist(), df["label"].tolist()
print(len(docs), "documents")

In [ ]:
from gensim.models import Word2Vec, FastText
from train_embeddings import PARAMS
MODELS = ROOT / "models"
cbow = Word2Vec.load(str(MODELS / "w2v_cbow.model")).wv
skip = Word2Vec.load(str(MODELS / "w2v_skipgram.model")).wv
ft   = FastText.load(str(MODELS / "fasttext.model")).wv
models = {"CBOW": cbow, "Skip-gram": skip, "fastText": ft}
print("Hyperparameters (same for all 3):", PARAMS)
for n, m in models.items(): print(f"{n:10s} vocab={len(m):,}  dim={m.vector_size}")

## B1/B2. Hyperparameters
| Parameter | Value | Why |
|---|---|---|
| vector_size | 100 | enough for a ~40k-article corpus, small to train |
| window | 5 | 5 words on each side as context |
| min_count | 5 | ignore very rare words (mostly typos / glued words) |
| epochs | 5 | gensim default |
| sg | 0 = CBOW, 1 = Skip-gram | the ONLY difference between B1 and B2 |
| fastText | sg=1, char n-grams 3-6, bucket=200,000 | same settings + sub-words (bucket keeps the saved file small) |

## B3. CBOW vs Skip-gram (edit in your own words)
- **CBOW** looks at the context words around a gap and predicts the **center** word (context -> center). It averages the context, so it is faster and smooth for frequent words.
- **Skip-gram** takes the **center** word and predicts each context word (center -> context). Every word gets more training updates, so it learns **rare words** better, but it is slower.
- Our expectation: Skip-gram should do better on our data, because Urdu news has many rare names and spellings, and our corpus is medium-sized.

## C1. Most similar words

In [ ]:
queries = ["پاکستان", "کرکٹ", "فلم", "معیشت", "موبائل"]   # change to words meaningful for you
for q in queries:
    print(f"\n=== {q} ===")
    for n, m in models.items():
        if q in m.key_to_index:
            print(f"{n:10s}", [w for w, s in m.most_similar(q, topn=6)])
        else:
            print(f"{n:10s} (not in vocabulary)")

**Comment (edit in your own words):** do the neighbours make sense? e.g. for کرکٹ we expect ٹیم، میچ، ورلڈ کپ, for فلم we expect اداکار، ڈرامہ.

## C2. Cosine similarity: expected similar vs unrelated

In [ ]:
pairs = [("کرکٹ", "میچ"), ("بینک", "قرض"), ("کرکٹ", "بجٹ"), ("فلم", "بینک")]
rows = []
for a, b in pairs:
    r = {"pair": f"{a} / {b}"}
    for n, m in models.items():
        r[n] = round(float(m.similarity(a, b)), 3) if a in m.key_to_index and b in m.key_to_index else None
    rows.append(r)
pd.DataFrame(rows)

**Comment (edit in your own words):** the first 2 pairs should have a higher similarity than the last 2.

## C3. Analogy: a - b + c = ?

In [ ]:
a, b, c = "لاہور", "پاکستان", "بھارت"     # Lahore - Pakistan + India = ? (expect Delhi / Mumbai)
for n, m in models.items():
    try:
        print(f"{n:10s}", m.most_similar(positive=[a, c], negative=[b], topn=5))
    except KeyError as e:
        print(f"{n:10s} missing word: {e}")

**Result (report honestly, even if it fails):** [write what you got]

## B4. fastText and rare / misspelled words
Word2Vec has one vector per WHOLE word, so a word it never saw has no vector at all.
fastText also learns vectors for character n-grams (3-6 letters) and builds a word from its pieces, so it can make a vector for a misspelled or unseen word.

In [ ]:
tests = ["پاکستانن", "کرکٹرز", "حکومتت"]     # misspelled / unseen forms
for w in tests:
    print(f"\n=== {w} ===")
    for n in ["CBOW", "Skip-gram"]:
        print(f"{n:10s}", "in vocab" if w in models[n].key_to_index else "NOT in vocabulary -> no vector")
    print("fastText  ", "in vocab" if w in ft.key_to_index else "not in vocab, but vector built from char n-grams:")
    print("           nearest:", [x for x, s in ft.most_similar(w, topn=5)])

**Comment (edit in your own words):** Word2Vec cannot do anything with the misspelled word, while fastText still finds the correct word as its nearest neighbour.